# 🔥 Chapter 4 — Autograd: Automatic Differentiation

---

### 🎯 What you'll learn
- How PyTorch tracks computations to compute gradients
- `requires_grad`, `.backward()`, `.grad`
- Gradient accumulation and zeroing
- `torch.no_grad()` and `detach()`
- ⚠️ Critical corner cases that cause silent bugs

## 4.1 — The Computation Graph

PyTorch builds a **dynamic computation graph** as you perform operations. When you call `.backward()`, it traverses this graph in reverse to compute gradients via the **chain rule**.

In [ ]:
import torch

# Create a tensor that tracks gradients
x = torch.tensor(3.0, requires_grad=True)

# Forward pass — build the computation graph
y = x ** 2 + 2 * x + 1   # y = x² + 2x + 1

# Backward pass — compute gradients
y.backward()

# dy/dx = 2x + 2 → at x=3: 2(3) + 2 = 8
print(f"x = {x}")
print(f"y = {y}")
print(f"dy/dx = {x.grad}")  # 8.0

## 4.2 — ⚠️ Corner Case #1: Gradient Accumulation

Gradients are **accumulated** (added) each time `.backward()` is called. If you don't zero them, they keep growing!

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

# First backward
y1 = x ** 2
y1.backward()
print(f"After 1st backward: grad = {x.grad}")  # 4.0 (2*2)

# ⚠️ Second backward WITHOUT zeroing
y2 = x ** 2
y2.backward()
print(f"After 2nd backward: grad = {x.grad}")  # 8.0! (accumulated: 4+4)

# ✅ Always zero gradients before backward
x.grad.zero_()
y3 = x ** 2
y3.backward()
print(f"After zeroing + backward: grad = {x.grad}")  # 4.0 ✅

## 4.3 — Disabling Gradient Tracking

Use `torch.no_grad()` during inference (prediction) to:
1. Save memory (no computation graph stored)
2. Speed up computation

In [ ]:
x = torch.randn(1000, 1000, requires_grad=True)

# ❌ This tracks gradients (wastes memory during inference)
y = x * 2 + 3

# ✅ Method 1: torch.no_grad() context manager
with torch.no_grad():
    y = x * 2 + 3
    print(f"requires_grad inside no_grad: {y.requires_grad}")  # False

# ✅ Method 2: detach() — creates a tensor sharing data but without grad
y_detached = x.detach()
print(f"Detached requires_grad: {y_detached.requires_grad}")  # False

# ⚠️ Corner Case: detach() shares memory!
y_detached[0, 0] = 999
print(f"Original modified? {x[0, 0]}")  # YES — shared memory!
# Use .detach().clone() for a truly independent copy

## 4.4 — ⚠️ Corner Case #2: Non-scalar Backward

`.backward()` only works directly on **scalar** tensors. For non-scalar outputs, you must provide a gradient argument.

In [ ]:
x = torch.tensor([1., 2., 3.], requires_grad=True)
y = x ** 2  # y is a vector, not a scalar!

# y.backward()  # ❌ RuntimeError: grad can be implicitly created only for scalar outputs

# ✅ Option 1: Reduce to scalar first
y.sum().backward()
print(f"Grad via sum: {x.grad}")  # [2, 4, 6]

# ✅ Option 2: Provide gradient tensor
x.grad.zero_()
y = x ** 2
y.backward(torch.ones_like(y))  # Equivalent to y.sum().backward()
print(f"Grad via ones: {x.grad}")  # [2, 4, 6]

## 4.5 — Practical Gradient Descent Example

In [ ]:
# Minimize f(x) = (x - 5)² using gradient descent
x = torch.tensor(0.0, requires_grad=True)
learning_rate = 0.1

print(f"{'Step':>4} | {'x':>8} | {'f(x)':>8} | {'grad':>8}")
print("-" * 40)

for step in range(20):
    # Forward
    f = (x - 5) ** 2
    
    # Backward
    f.backward()
    
    # Update (must be inside no_grad!)
    with torch.no_grad():
        x -= learning_rate * x.grad
    
    # Zero gradients!
    x.grad.zero_()
    
    if step % 4 == 0:
        print(f"{step:4d} | {x.item():8.4f} | {f.item():8.4f} | {x.grad.item() if x.grad is not None else 0:8.4f}")

print(f"\nFinal x = {x.item():.6f} (should approach 5.0)")

## ✏️ Exercises

In [ ]:
# Exercise 1: Compute the gradient of f(x, y) = x²y + y³ at x=2, y=3
# Expected: df/dx = 2xy = 12, df/dy = x² + 3y² = 31
# YOUR CODE HERE


# Exercise 2: Use gradient descent to find the minimum of f(x) = x⁴ - 4x² + 1
# Start from x = 3.0, learning rate = 0.01, 500 steps
# YOUR CODE HERE
